In [1]:
!pip install transformers datasets huggingface_hub transformers[torch] accelerate --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

In [3]:
from huggingface_hub import login

login()

In [4]:
import re
from sklearn.model_selection import train_test_split

In [5]:
f = open("./drive/MyDrive/metriccoders_datasets/history_of_kadambas.txt", "r")
text = f.readlines()

In [6]:
text

['\n',
 'Main menu\n',
 '\n',
 'WikipediaThe Free Encyclopedia\n',
 'Search Wikipedia\n',
 'Search\n',
 'Create account\n',
 'Log in\n',
 '\n',
 'Personal tools\n',
 'Contents hide\n',
 '(Top)\n',
 'History\n',
 'Toggle History subsection\n',
 'Origin\n',
 'Birth of Kingdom\n',
 'Expansion\n',
 'Decline\n',
 'Administration\n',
 'Economy\n',
 'Culture\n',
 'Toggle Culture subsection\n',
 'Religion\n',
 'Society\n',
 'Architecture\n',
 'Language\n',
 'In modern times\n',
 'See also\n',
 'Notes\n',
 'References\n',
 'Toggle References subsection\n',
 'Book\n',
 'Web\n',
 'External links\n',
 'Kadamba dynasty\n',
 '\n',
 'Article\n',
 'Talk\n',
 'Read\n',
 'Edit\n',
 'View history\n',
 '\n',
 'Tools\n',
 'Appearance hide\n',
 'Text\n',
 '\n',
 'Small\n',
 '\n',
 'Standard\n',
 '\n',
 'Large\n',
 'Width\n',
 '\n',
 'Standard\n',
 '\n',
 'Wide\n',
 'Color (beta)\n',
 '\n',
 'Automatic\n',
 '\n',
 'Light\n',
 '\n',
 'Dark\n',
 'Report an issue with dark mode\n',
 'From Wikipedia, the free en

In [7]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

Train dataset length: 352
Test dataset length: 63


In [8]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [10]:
from transformers import TextDataset, DataCollatorForLanguageModeling
model = AutoModelForCausalLM.from_pretrained("gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
def load_dataset(train_path, test_path, tokeinzer):
  train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=64)
  test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=64)
  data_collator = DataCollatorForLanguageModeling(
          tokenizer=tokenizer, mlm=False,
  )
  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (12795 > 1024). Running this sequence through the model will result in indexing errors


In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-history-of-kadambas",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_steps=400,
    save_steps=100,
    save_total_limit=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [13]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=14, training_loss=4.3299247196742465, metrics={'train_runtime': 10.7188, 'train_samples_per_second': 37.131, 'train_steps_per_second': 1.306, 'total_flos': 12999278592000.0, 'train_loss': 4.3299247196742465, 'epoch': 2.0})

In [14]:
trainer.save_model()

In [15]:
input_text = "Kadamba Dynasty was "
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

In [16]:
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [17]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Kadamba Dynasty was  the first dynasty of the  Kadamba dynasty  (Kadamba Dynasty  (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (Kadamba Dynasty (


In [18]:
from huggingface_hub import notebook_login, create_repo, Repository
notebook_login()


In [21]:
repo_name = "fine-tuned-gpt2-history-of-kadambas-updated"  # Change this to your desired repository name
from huggingface_hub import HfApi

# Initialize the HfApi instance
api = HfApi(token="")

# Create a new repository
username = api.whoami()['name']  # Get your Hugging Face username
full_repo_name = f"{username}/{repo_name}"

# Create the repository (you can also create it on the Hugging Face website)
api.create_repo(repo_name, private=False)

api.upload_folder(
    folder_path='./gpt2-history-of-kadambas',  # Path to the folder with your model
    repo_id=full_repo_name,  # Model repository name
    commit_message="GPT-2 Kadambas updated"
)

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

optimizer.pt:   0%|          | 0.00/996M [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Upload 8 LFS files:   0%|          | 0/8 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

events.out.tfevents.1724323398.c9811b1b06d1.426.0:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/metriccoders/fine-tuned-gpt2-history-of-kadambas-updated/commit/ddd501f2aba1ac18879f1dce4dbb3c019853b020', commit_message='GPT-2 Kadambas updated', commit_description='', oid='ddd501f2aba1ac18879f1dce4dbb3c019853b020', pr_url=None, pr_revision=None, pr_num=None)